### Import Dependencies

In [191]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [223]:
import os
os.chdir(r'C:\Kaggle_Competition\Playground\S6E5-F1-Pit-Stops')
from pathlib import Path
import numpy as np
import pandas as pd
import gc
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.inspection import permutation_importance
import mlflow
import shap
from tqdm.notebook import tqdm
import optuna

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from lightgbm import LGBMClassifier
from lightgbm import early_stopping
from xgboost import XGBClassifier
from xgboost.callback import EarlyStopping
from catboost import CatBoostClassifier
from pytabkit import RealMLP_TD_Classifier, TabM_D_Classifier, Resnet_RTDL_D_Classifier, TabM_HPO_Classifier
from sklearn.manifold import TSNE

# Preprocess
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import TargetEncoder, OneHotEncoder, StandardScaler, LabelEncoder, OrdinalEncoder, KBinsDiscretizer
from feature_engine.outliers import Winsorizer
from category_encoders import CountEncoder

# Externals
from src.utils import *
from src.fe import *
import config
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

### Prepare Datasets

In [177]:
train = read_csv_file('data/raw/train.csv')
test = read_csv_file('data/raw/test.csv')
orig = read_csv_file('data/orig/f1_strategy_dataset_v4.csv')

train = reduce_mem_usage(train)
test = reduce_mem_usage(test)
orig = reduce_mem_usage(orig)

train_ids = train[config.ID_COL].values
test_ids = test[config.ID_COL].values

print(f'Shape of train data: {train.shape}')
print(f'Shape of test data: {test.shape}')
print(f'Shape of orig data: {orig.shape}')

Shape of train data: (439140, 16)
Shape of test data: (188165, 15)
Shape of orig data: (101371, 16)


### Logistic Regression

In [ ]:
# Data
X = train.drop(['id', config.TARGET], axis=1)
y = train[config.TARGET].values
X_test = test.drop('id', axis=1)

In [ ]:
# pre-season testing flag
X['is_test_race'] = (X.race=='Pre-Season Testing').astype(int)
X_test['is_test_race'] = (X_test.race=='Pre-Season Testing').astype(int)

# 2023 year flag
X['is_2023'] = (X.year==2023).astype(int)
X_test['is_2023'] = (X_test.year==2023).astype(int)

In [ ]:
TE_columns = []

columns = config.BASE_CAT_COLS + config.BASE_NUM_COLS

for col1, col2 in tqdm(
    itertools.combinations(columns, 2),
    colour='green',
    ncols=300,
    desc="Interaction features",
    total=91
):

    name = f"{col1}_{col2}"

    train_combo = X[col1].astype(str) + "_" + X[col2].astype(str)
    test_combo = X_test[col1].astype(str) + "_" + X_test[col2].astype(str)

    combined = pd.concat([train_combo, test_combo], ignore_index=True)

    codes, _ = pd.factorize(combined)

    n_unique = pd.Series(codes).nunique()

    if n_unique > len(combined) // 2:
        continue

    X[name] = codes[:len(X)].astype("int32")
    X_test[name] = codes[len(X):].astype("int32")

    TE_columns.append(name)

Interaction features:   0%|                                                                                   …

In [ ]:
tmp_cols = X.columns

In [ ]:
# Predictions
oof_predictions = np.zeros(len(train))
test_predictions = np.zeros(len(X_test))

# Experiment Setup
EXPERIMENT_NAME = "LR"
VERSION = "v3"
RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# CV
skf = StratifiedKFold(
    n_splits=config.N_FOLDS,
    shuffle=True,
    random_state=config.SEED
)

fold_loglosses = []
fold_aucs = []

with mlflow.start_run(run_name=RUN_NAME):

    for fold, (train_indices, valid_indices) in enumerate(
        skf.split(X, y),
        start=1
    ):

        X_train = X.iloc[train_indices].copy()
        X_valid = X.iloc[valid_indices].copy()
        X_test_copy = X_test.copy()

        y_train = y[train_indices]
        y_valid = y[valid_indices]

        # Make TE columns consistent
        X_train[tmp_cols] = X_train[tmp_cols].astype(str).fillna("missing")
        X_valid[tmp_cols] = X_valid[tmp_cols].astype(str).fillna("missing")
        X_test_copy[tmp_cols] = X_test_copy[tmp_cols].astype(str).fillna("missing")

        # Target Encoding
        TE = TargetEncoder(
            cv=config.N_FOLDS,
            smooth='auto',
            shuffle=True,
            random_state=config.SEED
        )

        X_train[tmp_cols] = TE.fit_transform(X_train[tmp_cols], y_train)
        X_valid[tmp_cols] = TE.transform(X_valid[tmp_cols])
        X_test_copy[tmp_cols] = TE.transform(X_test_copy[tmp_cols])

        # Scaling
        scale_cols = tmp_cols

        scaler = StandardScaler()

        X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
        X_valid[scale_cols] = scaler.transform(X_valid[scale_cols])
        X_test_copy[scale_cols] = scaler.transform(X_test_copy[scale_cols])

        model = LogisticRegression(**config.LR_PARAMS)

        model.fit(X_train, y_train)

        fold_oof_predictions = model.predict_proba(X_valid)[:, 1]
        fold_test_predictions = model.predict_proba(X_test_copy)[:, 1]

        oof_predictions[valid_indices] = fold_oof_predictions
        test_predictions += fold_test_predictions / config.N_FOLDS

        fold_logloss = log_loss(y_valid, fold_oof_predictions)
        fold_auc = roc_auc_score(y_valid, fold_oof_predictions)

        fold_loglosses.append(fold_logloss)
        fold_aucs.append(fold_auc)

        print("=" * 50)
        print(f"Fold {fold} LogLoss : {fold_logloss:.5f}")
        print(f"Fold {fold} ROC AUC : {fold_auc:.5f}")
        print("=" * 50)

    oof_logloss = log_loss(y, oof_predictions)
    oof_auc = roc_auc_score(y, oof_predictions)

    std_oof_logloss = np.std(fold_loglosses)
    std_oof_auc = np.std(fold_aucs)

    mlflow.log_metric("oof_logloss", oof_logloss)
    mlflow.log_metric("oof_auc", oof_auc)
    mlflow.log_metric("std_oof_logloss", std_oof_logloss)
    mlflow.log_metric("std_oof_auc", std_oof_auc)

    mlflow.log_params(config.LR_PARAMS)

# Saving predictions
oof_proba_df = pd.DataFrame({
    'id': train_ids,
    'PitNextLap': oof_predictions
})

test_proba_df = pd.DataFrame({
    'id': test_ids,
    'PitNextLap': test_predictions
})

save_csv_file(
    oof_proba_df,
    Path(config.OOF_PROBA_DIR, f'{RUN_NAME}_oof_proba.csv')
)

save_csv_file(
    test_proba_df,
    Path(config.TEST_PROBA_DIR, f'{RUN_NAME}_test_proba.csv')
)

Fold 1 LogLoss : 0.34447
Fold 1 ROC AUC : 0.93268
Fold 2 LogLoss : 0.34472
Fold 2 ROC AUC : 0.93030
Fold 3 LogLoss : 0.34432
Fold 3 ROC AUC : 0.93167
Fold 4 LogLoss : 0.34456
Fold 4 ROC AUC : 0.93108
Fold 5 LogLoss : 0.34052
Fold 5 ROC AUC : 0.93219


### HistGradientBoosting

In [176]:
# Data
X = train.drop(['id', config.TARGET], axis=1)
y = train[config.TARGET].values
X_test = test.drop('id', axis=1)

In [180]:
# Predictions
oof_predictions = np.zeros(len(train))
test_predictions = np.zeros(len(X_test))

# Experiment Setup
EXPERIMENT_NAME = "HISTGBM"
VERSION = "v2"
RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}"

mlflow.set_experiment(EXPERIMENT_NAME)

# CV
skf = StratifiedKFold(
    n_splits=config.N_FOLDS,
    shuffle=True,
    random_state=config.SEED
)

fold_loglosses = []
fold_aucs = []

with mlflow.start_run(run_name=RUN_NAME):

    for fold, (train_indices, valid_indices) in enumerate(
        skf.split(X, y),
        start=1
    ):
        X_train = X.iloc[train_indices]
        X_valid = X.iloc[valid_indices]

        y_train = y[train_indices]
        y_valid = y[valid_indices]

        preprocessor = ColumnTransformer(
            transformers=[
                ('encoder', TargetEncoder(target_type='binary', cv=5, shuffle=True, random_state=config.SEED), tmp_cols)
            ],
            remainder='passthrough',
            n_jobs=-1
        )

        X_train_proc = preprocessor.fit_transform(X_train, y_train)
        X_valid_proc = preprocessor.transform(X_valid)
        X_test_proc = preprocessor.transform(X_test)

        model = HistGradientBoostingClassifier(**config.HISTGBM_PARAMS)

        model.fit(X_train_proc, y_train)

        fold_oof_predictions = model.predict_proba(X_valid_proc)[:, 1]
        fold_test_predictions = model.predict_proba(X_test_proc)[:, 1]

        oof_predictions[valid_indices] = fold_oof_predictions
        test_predictions += fold_test_predictions / config.N_FOLDS

        fold_logloss = log_loss(y_valid, fold_oof_predictions)
        fold_auc = roc_auc_score(y_valid, fold_oof_predictions)

        fold_loglosses.append(fold_logloss)
        fold_aucs.append(fold_auc)

        print("=" * 50)
        print(f"Fold {fold} LogLoss : {fold_logloss:.5f}")
        print(f"Fold {fold} ROC AUC : {fold_auc:.5f}")
        print("=" * 50)

    oof_logloss = log_loss(y, oof_predictions)
    oof_auc = roc_auc_score(y, oof_predictions)

    std_oof_logloss = np.std(fold_loglosses)
    std_oof_auc = np.std(fold_aucs)

    mlflow.log_metric("oof_logloss", oof_logloss)
    mlflow.log_metric("oof_auc", oof_auc)
    mlflow.log_metric("std_oof_logloss", std_oof_logloss)
    mlflow.log_metric("std_oof_auc", std_oof_auc)

    mlflow.log_params(config.HISTGBM_PARAMS) # Logging params

# Saving predictions
oof_proba_df = pd.DataFrame({'id': train_ids, 'PitNextLap': oof_predictions})
test_proba_df = pd.DataFrame({'id': test_ids, 'PitNextLap': test_predictions})

save_csv_file(oof_proba_df, Path(config.OOF_PROBA_DIR, f'{RUN_NAME}_oof_proba.csv'))
save_csv_file(test_proba_df, Path(config.TEST_PROBA_DIR, f'{RUN_NAME}_test_proba.csv'))

KeyboardInterrupt: 

### LGBM

In [ ]:
# Data
train_features = train.drop(['id', config.TARGET], axis=1)
target = train[config.TARGET].values
X_test = test.drop('id', axis=1)
orig_features = orig.drop([config.TARGET, 'normalized_tyrelife'], axis=1)

In [ ]:
# Winsorization
winsor = Winsorizer(
    capping_method='iqr',
    tail='both',
    fold=5,
    variables = [
    "tyrelife",
    "laptime (s)",
    "laptime_delta",
    "cumulative_degradation",
    "raceprogress"
    ]
)

train_features = winsor.fit_transform(train_features)
X_test = winsor.transform(X_test)
orig_features = winsor.transform(orig_features)

In [ ]:
# Frequency Encoding
for col in config.BASE_CAT_COLS:

    freq_map = train_features[col].value_counts(normalize=True).to_dict()

    train_features[f"{col}_freq"] = train_features[col].map(freq_map)
    X_test[f"{col}_freq"] = X_test[col].map(freq_map).fillna(0)
    orig_features[f"{col}_freq"] = orig_features[col].map(freq_map).fillna(0)

# Ordinal Encoding
encoder = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

train_features[config.BASE_CAT_COLS] = encoder.fit_transform(
    train_features[config.BASE_CAT_COLS].astype(str)
).astype("int32")

X_test[config.BASE_CAT_COLS] = encoder.transform(
    X_test[config.BASE_CAT_COLS].astype(str)
).astype("int32")

orig_features[config.BASE_CAT_COLS] = encoder.transform(
    orig_features[config.BASE_CAT_COLS].astype(str)
).astype("int32")

In [ ]:
# Predictions
oof_predictions = np.zeros(len(train))
test_predictions = np.zeros(len(X_test))

# Experiment Setup
EXPERIMENT_NAME = "LGBM"
VERSION = "v7"
RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# CV
skf = StratifiedKFold(
    n_splits=config.N_FOLDS,
    shuffle=True,
    random_state=config.SEED
)

fold_loglosses = []
fold_aucs = []

with mlflow.start_run(run_name=RUN_NAME):

    for fold, (train_indices, valid_indices) in enumerate(
        skf.split(train_features, target),
        start=1
    ):
        X_train = train_features.iloc[train_indices]
        X_valid = train_features.iloc[valid_indices]

        y_train = target[train_indices]
        y_valid = target[valid_indices]

        preprocessor = ColumnTransformer(
            transformers=[
                #('encoder', TargetEncoder(target_type='binary', cv=5, shuffle=True, random_state=config.SEED), config.BASE_CAT_COLS)
            ],
            remainder='passthrough',
            n_jobs=-1
        )

        X_train_proc = preprocessor.fit_transform(X_train, y_train)
        X_valid_proc = preprocessor.transform(X_valid)
        X_test_proc = preprocessor.transform(X_test)

        model = LGBMClassifier(**config.LGBM_PARAMS)

        model.fit(X_train_proc, y_train, eval_set=[(X_valid_proc, y_valid)], callbacks=[early_stopping(100)])

        fold_oof_predictions = model.predict_proba(X_valid_proc)[:, 1]
        fold_test_predictions = model.predict_proba(X_test_proc)[:, 1]

        oof_predictions[valid_indices] = fold_oof_predictions
        test_predictions += fold_test_predictions / config.N_FOLDS

        fold_logloss = log_loss(y_valid, fold_oof_predictions)
        fold_auc = roc_auc_score(y_valid, fold_oof_predictions)

        fold_loglosses.append(fold_logloss)
        fold_aucs.append(fold_auc)

        print("=" * 50)
        print(f"Fold {fold} LogLoss : {fold_logloss:.5f}")
        print(f"Fold {fold} ROC AUC : {fold_auc:.5f}")
        print("=" * 50)

    oof_logloss = log_loss(target, oof_predictions)
    oof_auc = roc_auc_score(target, oof_predictions)

    std_oof_logloss = np.std(fold_loglosses)
    std_oof_auc = np.std(fold_aucs)

    mlflow.log_metric("oof_logloss", oof_logloss)
    mlflow.log_metric("oof_auc", oof_auc)
    mlflow.log_metric("std_oof_logloss", std_oof_logloss)
    mlflow.log_metric("std_oof_auc", std_oof_auc)

    mlflow.log_params(config.LGBM_PARAMS) # Logging params

# Saving predictions
oof_proba_df = pd.DataFrame({'id': train_ids, 'PitNextLap': oof_predictions})
test_proba_df = pd.DataFrame({'id': test_ids, 'PitNextLap': test_predictions})

save_csv_file(oof_proba_df, Path(config.OOF_PROBA_DIR, f'{RUN_NAME}_oof_proba.csv'))
save_csv_file(test_proba_df, Path(config.TEST_PROBA_DIR, f'{RUN_NAME}_test_proba.csv'))

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[8930]	valid_0's auc: 0.949537
Fold 1 LogLoss : 0.28349
Fold 1 ROC AUC : 0.94954
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[9283]	valid_0's auc: 0.949361
Fold 2 LogLoss : 0.28083
Fold 2 ROC AUC : 0.94936
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[8355]	valid_0's auc: 0.948353
Fold 3 LogLoss : 0.28543
Fold 3 ROC AUC : 0.94835
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[7828]	valid_0's auc: 0.947556
Fold 4 LogLoss : 0.28808
Fold 4 ROC AUC : 0.94756
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[9529]	valid_0's auc: 0.948846
Fold 5 LogLoss : 0.28364
Fold 5 ROC AUC : 0.94885


In [ ]:
"""
==================================================
Fold 1 LogLoss : 0.28236
Fold 1 ROC AUC : 0.94970
==================================================
Fold 2 LogLoss : 0.28547
Fold 2 ROC AUC : 0.94810
==================================================
Fold 3 LogLoss : 0.28435
Fold 3 ROC AUC : 0.94873
==================================================
Fold 4 LogLoss : 0.28671
Fold 4 ROC AUC : 0.94782
==================================================
Fold 5 LogLoss : 0.28058
Fold 5 ROC AUC : 0.94934
=================================================="""

### XGBoost

In [234]:
# Data
X = train.drop(['id', config.TARGET], axis=1)
y = train[config.TARGET]
X_test = test.drop('id', axis=1)

X_orig = orig.dropna(axis=0)
y_orig = X_orig[config.TARGET]
X_orig = X_orig.drop([config.TARGET], axis=1)

In [235]:
# parameters: {'subsample': 0.9895883554904479, 'colsample_bytree': 0.5643780817820577, 'reg_alpha': 3.318715714208402, 'reg_lambda': 0.023048062755762426, 'gamma': 0.008859306810836383}. Best is trial 42 with value: 0.9486474315931364.

In [236]:
# pre-season testing flag
train_features['is_test_race'] = (train_features.race=='Pre-Season Testing').astype(int)
X_test['is_test_race'] = (X_test.race=='Pre-Season Testing').astype(int)

# 2023 year flag
train_features['is_2023'] = (train_features.year==2023).astype(int)
X_test['is_2023'] = (X_test.year==2023).astype(int)

In [237]:
stratify_labels = (
    target.astype(str) + "_" +
    train_features["is_test_race"].astype(str)
)

In [238]:
# Arithmetic interaction
for df in [train_features, X_test]:
    df['_LapNumber_/_RaceProgress'] = (df['lapnumber'] / (df['raceprogress'] + 1e-6)).astype('float32')
    df['_TyreLife_/_LapNumber'] = (df['tyrelife'] / df['lapnumber'].clip(lower=1)).astype('float32')
    df['_LapTime (s)_*_Cumulative_Degradation'] = (df['laptime (s)'] * df['cumulative_degradation']).astype('float32')
    df['_LapTime (s)_*_Cumulative_Degradation_abs'] = (df['laptime (s)'] * df['cumulative_degradation'].abs()).astype('float32')
    df['_LapTime (s)_/_Cumulative_Degradation_abs'] = (df['laptime (s)'] / (df['cumulative_degradation'].abs() + 1e-6)).astype('float32')

In [239]:
# Create n-gram categorical features
train_features, bi_cols = create_cat_bigrams(
    train_features,
    config.BASE_CAT_COLS
)

train_features, tri_cols = create_cat_trigrams(
    train_features,
    config.BASE_CAT_COLS
)

X_test, _ = create_cat_bigrams(X_test, config.BASE_CAT_COLS)
X_test, _ = create_cat_trigrams(X_test, config.BASE_CAT_COLS)

orig_features, _ = create_cat_bigrams(orig_features, config.BASE_CAT_COLS)
orig_features, _ = create_cat_trigrams(orig_features, config.BASE_CAT_COLS)

# All generated categorical columns
all_ngram_cols = bi_cols + tri_cols

for col in all_ngram_cols:

    combined = pd.concat(
        [
            train_features[col],
            X_test[col],
            orig_features[col]
        ]
    ).astype(str)

    codes, _ = pd.factorize(combined)

    n_train = len(train_features)
    n_test = len(X_test)

    train_features[col] = codes[:n_train].astype("int32")
    X_test[col] = codes[n_train:n_train+n_test].astype("int32")
    orig_features[col] = codes[n_train+n_test:].astype("int32")

Added 3 bigram features
Added 1 trigram features
Added 3 bigram features
Added 1 trigram features
Added 3 bigram features
Added 1 trigram features


In [240]:
for col in config.BASE_CAT_COLS:

    combined = pd.concat(
        [
            train_features[col].astype(str),
            X_test[col].astype(str),
            orig_features[col].astype(str)
        ]
    )

    codes, _ = pd.factorize(combined)

    n_train = len(train_features)
    n_test = len(X_test)

    train_features[col] = pd.Categorical(codes[:n_train])
    X_test[col] = pd.Categorical(codes[n_train:n_train+n_test])
    orig_features[col] = pd.Categorical(codes[n_train+n_test:])

In [241]:
TE_COLS = all_ngram_cols + config.BASE_CAT_COLS

In [242]:
tmp_num_cols = [
    'lapnumber',
    'stint',
    'tyrelife',
    'position',
    'laptime (s)',
    'laptime_delta',
    'cumulative_degradation',
    'raceprogress',
    'position_change'
    ]

In [243]:
train_features, bins_cols = create_qcut_bins(train_features, tmp_num_cols)
X_test, _ = create_qcut_bins(X_test, tmp_num_cols)

Added 9 qcut features
Added 9 qcut features


In [244]:
# Predictions
oof_predictions = np.zeros(len(train))
test_predictions = np.zeros(len(X_test))

# Experiment Setup
EXPERIMENT_NAME = "XGB"
VERSION = "v14"
RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# CV
skf = StratifiedKFold(
    n_splits=config.N_FOLDS,
    shuffle=True,
    random_state=config.SEED
)

fold_loglosses = []
fold_aucs = []

with mlflow.start_run(run_name=RUN_NAME):

    for fold, (train_indices, valid_indices) in enumerate(
        skf.split(train_features, stratify_labels),
        start=1
    ):
        X_train = train_features.iloc[train_indices]
        X_valid = train_features.iloc[valid_indices]

        y_train = target[train_indices]
        y_valid = target[valid_indices]

        preprocessor = ColumnTransformer(
            transformers=[
                ('encoder', TargetEncoder(cv=5, target_type='binary', shuffle=True, random_state=config.SEED), TE_COLS + bins_cols)
            ],
            remainder='passthrough',
            n_jobs=-1
        )

        X_train_proc = preprocessor.fit_transform(X_train, y_train)
        X_valid_proc = preprocessor.transform(X_valid)
        X_test_proc = preprocessor.transform(X_test)

        early_stop = EarlyStopping(rounds=100, metric_name='auc', save_best=True)
        model = XGBClassifier(**config.XGBM_PARAMS)
        model.fit(X_train_proc, y_train, eval_set=[(X_valid_proc, y_valid)], verbose=1000)

        fold_oof_predictions = model.predict_proba(X_valid_proc)[:, 1]
        fold_test_predictions = model.predict_proba(X_test_proc)[:, 1]

        oof_predictions[valid_indices] = fold_oof_predictions
        test_predictions += fold_test_predictions / config.N_FOLDS

        fold_logloss = log_loss(y_valid, fold_oof_predictions)
        fold_auc = roc_auc_score(y_valid, fold_oof_predictions)

        fold_loglosses.append(fold_logloss)
        fold_aucs.append(fold_auc)

        print("=" * 50)
        print(f"Fold {fold} LogLoss : {fold_logloss:.5f}")
        print(f"Fold {fold} ROC AUC : {fold_auc:.5f}")
        print("=" * 50)

    oof_logloss = log_loss(target, oof_predictions)
    oof_auc = roc_auc_score(target, oof_predictions)

    std_oof_logloss = np.std(fold_loglosses)
    std_oof_auc = np.std(fold_aucs)

    mlflow.log_metric("oof_logloss", oof_logloss)
    mlflow.log_metric("oof_auc", oof_auc)
    mlflow.log_metric("std_oof_logloss", std_oof_logloss)
    mlflow.log_metric("std_oof_auc", std_oof_auc)

    mlflow.log_params(config.XGBM_PARAMS) # Logging params

# Saving predictions
oof_proba_df = pd.DataFrame({'id': train_ids, 'PitNextLap': oof_predictions})
test_proba_df = pd.DataFrame({'id': test_ids, 'PitNextLap': test_predictions})

save_csv_file(oof_proba_df, Path(config.OOF_PROBA_DIR, f'{RUN_NAME}_oof_proba.csv'))
save_csv_file(test_proba_df, Path(config.TEST_PROBA_DIR, f'{RUN_NAME}_test_proba.csv'))

[0]	validation_0-auc:0.91178
[1000]	validation_0-auc:0.94549
[2000]	validation_0-auc:0.94876
[3000]	validation_0-auc:0.95011
[4000]	validation_0-auc:0.95080
[5000]	validation_0-auc:0.95122
[6000]	validation_0-auc:0.95145
[7000]	validation_0-auc:0.95159
[8000]	validation_0-auc:0.95167
[9000]	validation_0-auc:0.95173
[9999]	validation_0-auc:0.95175
Fold 1 LogLoss : 0.22002
Fold 1 ROC AUC : 0.95176
[0]	validation_0-auc:0.90852
[1000]	validation_0-auc:0.94378
[2000]	validation_0-auc:0.94711
[3000]	validation_0-auc:0.94836
[4000]	validation_0-auc:0.94909
[5000]	validation_0-auc:0.94949
[6000]	validation_0-auc:0.94979
[7000]	validation_0-auc:0.94996
[8000]	validation_0-auc:0.95006
[9000]	validation_0-auc:0.95010
[9457]	validation_0-auc:0.95011
Fold 2 LogLoss : 0.22339
Fold 2 ROC AUC : 0.95011
[0]	validation_0-auc:0.91139
[1000]	validation_0-auc:0.94520
[2000]	validation_0-auc:0.94813
[3000]	validation_0-auc:0.94936
[4000]	validation_0-auc:0.95006
[5000]	validation_0-auc:0.95045
[6000]	valida

### Resnet

In [ ]:
# Data
X = train.drop(['id', config.TARGET], axis=1)
y = train[config.TARGET].values
X_test = test.drop('id', axis=1)

orig_features = orig.drop([config.TARGET], axis=1)
orig_features = orig_features.dropna(axis=0)

In [ ]:
# Predictions
oof_predictions = np.zeros(len(train))
test_predictions = np.zeros(len(X_test))

# Experiment Setup
EXPERIMENT_NAME = "RESNET"
VERSION = "v2"
RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# CV
skf = StratifiedKFold(
    n_splits=config.N_FOLDS,
    shuffle=True,
    random_state=config.SEED
)

fold_loglosses = []
fold_aucs = []

with mlflow.start_run(run_name=RUN_NAME):

    for fold, (train_indices, valid_indices) in enumerate(
        skf.split(X, y),
        start=1
    ):
        X_train = X.iloc[train_indices]
        X_valid = X.iloc[valid_indices]

        y_train = y[train_indices]
        y_valid = y[valid_indices]
        X_test_copy = X_test.copy()

        preprocessor = ColumnTransformer(
            transformers=[
                ('encoder', TargetEncoder(cv=5, target_type='binary', shuffle=True, random_state=config.SEED), config.BASE_CAT_COLS)
            ],
            remainder='passthrough',
            n_jobs=-1
        )

        X_train_proc = preprocessor.fit_transform(X_train, y_train)
        X_valid_proc = preprocessor.transform(X_valid)
        X_test_proc = preprocessor.transform(X_test_copy)

        model = Resnet_RTDL_D_Classifier(**config.Resnet_PARAMS)
        model.fit(X_train_proc, y_train, X_valid_proc, y_valid)

        fold_oof_predictions = model.predict_proba(X_valid_proc)[:, 1]
        fold_test_predictions = model.predict_proba(X_test_proc)[:, 1]

        oof_predictions[valid_indices] = fold_oof_predictions
        test_predictions += fold_test_predictions / config.N_FOLDS

        fold_logloss = log_loss(y_valid, fold_oof_predictions)
        fold_auc = roc_auc_score(y_valid, fold_oof_predictions)

        fold_loglosses.append(fold_logloss)
        fold_aucs.append(fold_auc)

        print("=" * 50)
        print(f"Fold {fold} LogLoss : {fold_logloss:.5f}")
        print(f"Fold {fold} ROC AUC : {fold_auc:.5f}")
        print("=" * 50)

    oof_logloss = log_loss(y, oof_predictions)
    oof_auc = roc_auc_score(y, oof_predictions)

    std_oof_logloss = np.std(fold_loglosses)
    std_oof_auc = np.std(fold_aucs)

    mlflow.log_metric("oof_logloss", oof_logloss)
    mlflow.log_metric("oof_auc", oof_auc)
    mlflow.log_metric("std_oof_logloss", std_oof_logloss)
    mlflow.log_metric("std_oof_auc", std_oof_auc)

    mlflow.log_params(config.Resnet_PARAMS) # Logging params

# Saving predictions
oof_proba_df = pd.DataFrame({'id': train_ids, 'PitNextLap': oof_predictions})
test_proba_df = pd.DataFrame({'id': test_ids, 'PitNextLap': test_predictions})

save_csv_file(oof_proba_df, Path(config.OOF_PROBA_DIR, f'{RUN_NAME}_oof_proba.csv'))
save_csv_file(test_proba_df, Path(config.TEST_PROBA_DIR, f'{RUN_NAME}_test_proba.csv'))

Columns classified as continuous: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86]
Columns classified as categorical: []
RTDL classifier
[('epoch_timer', <skorch.callbacks.logging.EpochTimer object at 0x0000016F299F0E90>), ('train_loss', <skorch.callbacks.scoring.PassthroughScoring object at 0x0000016F39A93B50>), ('valid_loss', <skorch.callbacks.scoring.PassthroughScoring object at 0x0000016F39A92A90>), ('valid_acc', <skorch.callbacks.scoring.EpochScoring object at 0x0000016F2F9CC8D0>), ('print_log', <skorch.callbacks.logging.PrintLog object at 0x0000016F2F9CC990>)]
Initializing UniquePrefixCheckpoint
fn_prefix is 1577098206416
Re-initializing module because the following parameters were re-set: modul

KeyboardInterrupt: 

### Realmlp

In [224]:
# Data
X = train.drop(['id', 'PitNextLap'], axis=1)
y = train['PitNextLap']
X_test = test.drop('id', axis=1)

orig = orig.dropna(axis=0)
y_orig = orig['PitNextLap']
orig = orig.drop(['PitNextLap', 'Normalized_TyreLife'], axis=1)

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()

In [225]:
category_map = {}
important_combos = [
    ('Race', 'Compound'),
    ('Race', 'Year'),
]

In [ ]:
def feature_engineering(df, fit=False):
    df = df.copy()

    new_cat_cols = []
    new_num_cols = []
    combo_names = []

    # Arithmetic interaction
    df['_LapNumber_/_RaceProgress'] = (df['LapNumber'] / (df['RaceProgress'] + 1e-6)).astype('float32')
    df['_TyreLife_/_LapNumber'] = (df['TyreLife'] / df['LapNumber'].clip(lower=1)).astype('float32')
    df['_LapTime (s)_*_Cumulative_Degradation'] = (df['LapTime (s)'] * df['Cumulative_Degradation']).astype('float32')
    df['_LapTime (s)_*_Cumulative_Degradation_abs'] = (df['LapTime (s)'] * df['Cumulative_Degradation'].abs()).astype('float32')
    df['_LapTime (s)_/_Cumulative_Degradation_abs'] = (df['LapTime (s)'] / (df['Cumulative_Degradation'].abs() + 1e-6)).astype('float32')

    # Categorize numericals
    for col in num_cols + ['_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber']:
        cat_name = f"{col}_cat_" if col in num_cols else f"{col[1:]}_cat_"
        floored = pd.Series(np.floor(df[col]), index=df.index)

        if fit:
            codes, uniques = pd.factorize(floored)
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = floored.map(code_map).fillna(-1).astype('int32')

        df[cat_name] = codes.astype(str)

    # Count encoding
    for col in cat_cols + ['Year_cat_', 'PitStop_cat_']:
        count_name = f"_{col}_count" if col in cat_cols else f"_{col[:-1]}_count"

        if fit:
            count_map = df[col].value_counts()
            category_map[count_name] = count_map
        else:
            count_map = category_map[count_name]

        df[count_name] = df[col].map(count_map).fillna(0).astype('int32')

    # Discretize numericals
    bin_config = {'RaceProgress': [200], 'LapTime (s)': [7]}
    for col, bins_list in bin_config.items():
        for n_bins in bins_list:
            strategy = 'quantile'
            bin_name = f"{col}_{n_bins}_{strategy}_bin_"

            if fit:
                kb = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy=strategy, subsample=None)
                binned = kb.fit_transform(df[[col]]).ravel().astype('int32')
                category_map[bin_name] = kb
            else:
                kb = category_map[bin_name]
                binned = kb.transform(df[[col]]).ravel().astype('int32')

            df[bin_name] = binned.astype(str)

    combo_names = []
    for cols in important_combos:
        combo_name = '_'.join(cols) + '_'
        combo_names.append(combo_name)
        combo_series = df[cols[0]].astype(str)
        for col in cols[1:]:
            combo_series = combo_series + '_' + df[col].astype(str)
        if fit:
            codes, uniques = pd.factorize(combo_series, sort=False)
            category_map[combo_name] = uniques
        else:
            uniques = category_map[combo_name]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = combo_series.map(code_map).fillna(-1).astype('int32')
        df[combo_name] = codes
        df[combo_name] = df[combo_name].astype(str)   

    new_cat_cols = [col for col in df.columns if col.endswith('_')]
    new_num_cols = [col for col in df.columns if col.startswith('_')]
    return df, new_cat_cols, new_num_cols, combo_names

X, new_cat_cols, new_num_cols, combo_names = feature_engineering(X, fit=True)
X_test, new_cat_cols, new_num_cols, combo_names = feature_engineering(X_test, fit=False)
orig, new_cat_cols, new_num_cols, combo_names = feature_engineering(orig, fit=False)
cat_cols += new_cat_cols; num_cols += new_num_cols
print("len(new_cat_cols):", len(new_cat_cols))
print("len(new_num_cols):", len(new_num_cols), "\n")

print("prep len(cat_cols):", len(cat_cols))
print("prep len(num_cols):", len(num_cols), "\n")
print("X prep shape:", X.shape)
print("X_test prep shape:", X_test.shape)
print("orig prep shape:", orig.shape, "\n")

In [ ]:
# Predictions
oof_predictions = np.zeros(len(X))
test_predictions = np.zeros(len(X_test))

# Experiment Setup
EXPERIMENT_NAME = "REALMLP"
VERSION = "v1"
RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# CV
skf = StratifiedKFold(
    n_splits=config.N_FOLDS,
    shuffle=True,
    random_state=config.SEED
)

fold_loglosses = []
fold_aucs = []

with mlflow.start_run(run_name=RUN_NAME):

    for fold, ((tr_idx, val_idx), (or_tr_idx, or_val_idx)) in enumerate(
        zip(skf.split(X, y), skf.split(orig, y_orig)), start=1
    ):
        X_tr = X.iloc[tr_idx].copy()
        orig_tr = orig.iloc[or_tr_idx].copy()
        X_tr = pd.concat([X_tr, orig_tr], axis=0).reset_index(drop=True)
        y_tr = pd.concat([y.iloc[tr_idx], y_orig.iloc[or_tr_idx]], axis=0).reset_index(drop=True)
        X_val = X.iloc[val_idx].copy()
        y_val = y.iloc[val_idx]
        X_tst = X_test.copy()

        # Target encoding
        te_cols = combo_names
        TE = TargetEncoder(cv=config.N_FOLDS, smooth='auto', shuffle=True, random_state=config.SEED)

        tr_enc = TE.fit_transform(X_tr[te_cols], y_tr)
        val_enc = TE.transform(X_val[te_cols])
        tst_enc = TE.transform(X_tst[te_cols])

        te_names = [f"_{col}TE" for col in te_cols]

        X_tr[te_names] = tr_enc
        X_val[te_names] = val_enc
        X_tst[te_names] = tst_enc

        model = RealMLP_TD_Classifier(**config.REALMLP_PARAMS)
        model.fit(X_tr, y_tr, X_val, y_val)

        val_preds = model.predict_proba(X_val)[:, 1]
        fold_test_preds = model.predict_proba(X_tst)[:, 1]

        oof_predictions[val_idx] = val_preds
        test_predictions += fold_test_preds / config.N_FOLDS

        fold_logloss = log_loss(y_val, val_preds)
        fold_auc = roc_auc_score(y_val, val_preds)

        fold_loglosses.append(fold_logloss)
        fold_aucs.append(fold_auc)

        mlflow.log_metric("fold_logloss", fold_logloss, step=fold)
        mlflow.log_metric("fold_auc", fold_auc, step=fold)

        print(f"Fold {fold} | LogLoss: {fold_logloss:.5f} | AUC: {fold_auc:.5f}")

    oof_logloss = log_loss(y, oof_predictions)
    oof_auc = roc_auc_score(y, oof_predictions)

    std_oof_logloss = np.std(fold_loglosses)
    std_oof_auc = np.std(fold_aucs)

    mlflow.log_metric("oof_logloss", oof_logloss)
    mlflow.log_metric("oof_auc", oof_auc)
    mlflow.log_metric("std_oof_logloss", std_oof_logloss)
    mlflow.log_metric("std_oof_auc", std_oof_auc)

    mlflow.log_params(config.REALMLP_PARAMS)

# Saving predictions
oof_proba_df = pd.DataFrame({'id': train_ids, 'PitNextLap': oof_predictions})
test_proba_df = pd.DataFrame({'id': test_ids, 'PitNextLap': test_predictions})

save_csv_file(oof_proba_df, Path(config.OOF_PROBA_DIR, f'{RUN_NAME}_oof_proba.csv'))
save_csv_file(test_proba_df, Path(config.TEST_PROBA_DIR, f'{RUN_NAME}_test_proba.csv'))

Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', '_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber', '_LapTime (s)_*_Cumulative_Degradation', '_LapTime (s)_*_Cumulative_Degradation_abs', '_LapTime (s)_/_Cumulative_Degradation_abs', '_Driver_count', '_Compound_count', '_Race_count', '_Year_cat_count', '_PitStop_cat_count', '_Race_Compound_TE', '_Race_Year_TE']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'Year_cat_', 'PitStop_cat_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_', 'LapNumber_/_RaceProgress_cat_', 'TyreLife_/_LapNumber_cat_', 'RaceProgress_200_quantile_bin_', 'LapTime (s)_7_quantile_bin_', 'Race_Compound_', 'Race_Year_']


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 1/6: val 1-auc_ovr = 0.059062
Epoch 2/6: val 1-auc_ovr = 0.050563
Epoch 3/6: val 1-auc_ovr = 0.047970
Epoch 4/6: val 1-auc_ovr = 0.046373
Epoch 5/6: val 1-auc_ovr = 0.044949
Epoch 6/6: val 1-auc_ovr = 0.044748


`Trainer.fit` stopped: `max_epochs=6` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Fold 1 | LogLoss: 0.21241 | AUC: 0.95525
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', '_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber', '_LapTime (s)_*_Cumulative_Degradation', '_LapTime (s)_*_Cumulative_Degradation_abs', '_LapTime (s)_/_Cumulative_Degradation_abs', '_Driver_count', '_Compound_count', '_Race_count', '_Year_cat_count', '_PitStop_cat_count', '_Race_Compound_TE', '_Race_Year_TE']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'Year_cat_', 'PitStop_cat_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_', 'LapNumber_/_RaceProgress_cat_', 'TyreLife_/_LapNumber_cat_', 'RaceProgress_200_quantile_bin_', 'LapTime (s)_7_quantile_bin_', 'Race_Compound_', 'Race_Year_']


...

Fold 4 | LogLoss: 0.21733 | AUC: 0.95327
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', '_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber', '_LapTime (s)_*_Cumulative_Degradation', '_LapTime (s)_*_Cumulative_Degradation_abs', '_LapTime (s)_/_Cumulative_Degradation_abs', '_Driver_count', '_Compound_count', '_Race_count', '_Year_cat_count', '_PitStop_cat_count', '_Race_Compound_TE', '_Race_Year_TE']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'Year_cat_', 'PitStop_cat_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_', 'LapNumber_/_RaceProgress_cat_', 'TyreLife_/_LapNumber_cat_', 'RaceProgress_200_quantile_bin_', 'LapTime (s)_7_quantile_bin_', 'Race_Compound_', 'Race_Year_']


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 1/6: val 1-auc_ovr = 0.060082
Epoch 2/6: val 1-auc_ovr = 0.050976
Epoch 3/6: val 1-auc_ovr = 0.048408
Epoch 4/6: val 1-auc_ovr = 0.046641
Epoch 5/6: val 1-auc_ovr = 0.045344
Epoch 6/6: val 1-auc_ovr = 0.045202


`Trainer.fit` stopped: `max_epochs=6` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Fold 5 | LogLoss: 0.21333 | AUC: 0.95480


### CatBoost

In [ ]:
# Data
X = train.drop(['id', config.TARGET], axis=1)
y = train[config.TARGET]
X_test = test.drop('id', axis=1)
orig = orig.dropna(axis=0)

X_orig = orig.drop([config.TARGET, 'normalized_tyrelife'], axis=1)
y_orig = orig[config.TARGET]

In [ ]:
TE_columns = []

columns = config.BASE_CAT_COLS + config.BASE_NUM_COLS

for col1, col2 in tqdm(
    itertools.combinations(columns, 2),
    colour='green',
    ncols=300,
    desc="Interaction features",
    total=91
):

    name = f"{col1}_{col2}"

    train_combo = X[col1].astype(str) + "_" + X[col2].astype(str)
    test_combo = X_test[col1].astype(str) + "_" + X_test[col2].astype(str)
    orig_combo = X_orig[col1].astype(str) + "_" + X_orig[col2].astype(str)

    combined = pd.concat(
        [train_combo, test_combo, orig_combo],
        ignore_index=True
    )

    codes, _ = pd.factorize(combined)

    n_unique = pd.Series(codes).nunique()

    if n_unique > len(combined) // 2:
        continue

    n_train = len(X)
    n_test = len(X_test)
    n_orig = len(X_orig)

    X[name] = codes[:n_train].astype("int32")

    X_test[name] = codes[
        n_train:n_train + n_test
    ].astype("int32")

    X_orig[name] = codes[
        n_train + n_test:
        n_train + n_test + n_orig
    ].astype("int32")

    TE_columns.append(name)

Interaction features:   0%|                                                                                   …

In [ ]:
# pre-season testing flag
X['is_test_race'] = (X.race=='Pre-Season Testing').astype(int)
X_test['is_test_race'] = (X_test.race=='Pre-Season Testing').astype(int)
X_orig['is_test_race'] = (X_orig.race=='Pre-Season Testing').astype(int)

# 2023 year flag
X['is_2023'] = (X.year==2023).astype(int)
X_test['is_2023'] = (X_test.year==2023).astype(int)
X_orig['is_2023'] = (X_orig.year==2023).astype(int)

In [ ]:
combo = pd.concat([X, X_orig], axis=0)

In [ ]:
M = combo[config.BASE_NUM_COLS].max()

In [ ]:
X = create_digit_features(X, M, config.BASE_NUM_COLS)
X_test = create_digit_features(X, M, config.BASE_NUM_COLS)

In [ ]:
X = X.drop(config.BASE_NUM_COLS, axis=1)
X_test = X_test.drop(config.BASE_NUM_COLS, axis=1)

In [ ]:
tmp_cols = X.columns

In [ ]:
# Predictions
oof_predictions = np.zeros(len(train))
test_predictions = np.zeros(len(X_test))

# Experiment Setup
EXPERIMENT_NAME = "CatGBM"
VERSION = "v5"
RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# CV
skf = StratifiedKFold(
    n_splits=config.N_FOLDS,
    shuffle=True,
    random_state=config.SEED
)

fold_loglosses = []
fold_aucs = []

with mlflow.start_run(run_name=RUN_NAME):

    for fold, (train_indices, valid_indices) in enumerate(
        skf.split(X, y),
        start=1
    ):

        X_train = X.iloc[train_indices].copy()
        X_valid = X.iloc[valid_indices].copy()
        X_tst = X_test.copy()

        y_train = y[train_indices]
        y_valid = y[valid_indices]

        # Target Encoding
        TE = TargetEncoder(
            cv=5,
            target_type='binary',
            shuffle=True,
            random_state=config.SEED
        )

        X_train[tmp_cols] = X_train[tmp_cols].astype(str)
        X_valid[tmp_cols] = X_valid[tmp_cols].astype(str)
        X_tst[tmp_cols] = X_tst[tmp_cols].astype(str)

        X_train[tmp_cols] = TE.fit_transform(X_train[tmp_cols], y_train)
        X_valid[tmp_cols] = TE.transform(X_valid[tmp_cols])
        X_tst[tmp_cols] = TE.transform(X_tst[tmp_cols])

        model = CatBoostClassifier(**config.CATGBM_PARAMS)

        model.fit(
            X_train,
            y_train,
            eval_set=(X_valid, y_valid),
            verbose_eval=500
        )

        fold_oof_predictions = model.predict_proba(X_valid)[:, 1]
        fold_test_predictions = model.predict_proba(X_tst)[:, 1]

        oof_predictions[valid_indices] = fold_oof_predictions
        test_predictions += fold_test_predictions / config.N_FOLDS

        fold_logloss = log_loss(y_valid, fold_oof_predictions)
        fold_auc = roc_auc_score(y_valid, fold_oof_predictions)

        fold_loglosses.append(fold_logloss)
        fold_aucs.append(fold_auc)

        print("=" * 50)
        print(f"Fold {fold} LogLoss : {fold_logloss:.5f}")
        print(f"Fold {fold} ROC AUC : {fold_auc:.5f}")
        print("=" * 50)

    oof_logloss = log_loss(y, oof_predictions)
    oof_auc = roc_auc_score(y, oof_predictions)

    std_oof_logloss = np.std(fold_loglosses)
    std_oof_auc = np.std(fold_aucs)

    mlflow.log_metric("oof_logloss", oof_logloss)
    mlflow.log_metric("oof_auc", oof_auc)
    mlflow.log_metric("std_oof_logloss", std_oof_logloss)
    mlflow.log_metric("std_oof_auc", std_oof_auc)

    mlflow.log_params(config.CATGBM_PARAMS)

# Saving predictions
oof_proba_df = pd.DataFrame({
    'id': train_ids,
    'PitNextLap': oof_predictions
})

test_proba_df = pd.DataFrame({
    'id': test_ids,
    'PitNextLap': test_predictions
})

save_csv_file(
    oof_proba_df,
    Path(config.OOF_PROBA_DIR, f'{RUN_NAME}_oof_proba.csv')
)

save_csv_file(
    test_proba_df,
    Path(config.TEST_PROBA_DIR, f'{RUN_NAME}_test_proba.csv')
)

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9171057	best: 0.9171057 (0)	total: 242ms	remaining: 40m 18s
500:	test: 0.9400820	best: 0.9400820 (500)	total: 8.35s	remaining: 2m 38s
1000:	test: 0.9431884	best: 0.9431884 (1000)	total: 19.1s	remaining: 2m 51s
1500:	test: 0.9447031	best: 0.9447031 (1500)	total: 29.4s	remaining: 2m 46s
2000:	test: 0.9456394	best: 0.9456396 (1999)	total: 39.1s	remaining: 2m 36s
2500:	test: 0.9463086	best: 0.9463086 (2500)	total: 48.8s	remaining: 2m 26s
3000:	test: 0.9467947	best: 0.9467947 (3000)	total: 58.5s	remaining: 2m 16s
3500:	test: 0.9471522	best: 0.9471522 (3500)	total: 1m 8s	remaining: 2m 6s
4000:	test: 0.9474232	best: 0.9474233 (3999)	total: 1m 17s	remaining: 1m 56s
4500:	test: 0.9476519	best: 0.9476519 (4500)	total: 1m 27s	remaining: 1m 46s
5000:	test: 0.9478381	best: 0.9478381 (5000)	total: 1m 36s	remaining: 1m 36s
5500:	test: 0.9479969	best: 0.9479969 (5500)	total: 1m 46s	remaining: 1m 26s
6000:	test: 0.9481176	best: 0.9481176 (6000)	total: 1m 55s	remaining: 1m 17s
6500:	test: 0.9

KeyboardInterrupt: 

### TABM

In [ ]:
# Predictions
oof_predictions = np.zeros(len(X))
test_predictions = np.zeros(len(X_test))

# Experiment Setup
EXPERIMENT_NAME = "TABM"
VERSION = "v2"
RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# CV
skf = StratifiedKFold(
    n_splits=config.N_FOLDS,
    shuffle=True,
    random_state=config.SEED
)

fold_loglosses = []
fold_aucs = []

with mlflow.start_run(run_name=RUN_NAME):

    for fold, ((tr_idx, val_idx), (or_tr_idx, or_val_idx)) in enumerate(

        zip(
            skf.split(X, y),
            skf.split(X_orig, y_orig)
        ),

        start=1
    ):

        # Main train/valid split
        X_tr = X.iloc[tr_idx].copy()
        X_val = X.iloc[val_idx].copy()

        y_tr = y.iloc[tr_idx].reset_index(drop=True)
        y_val = y.iloc[val_idx].reset_index(drop=True)

        # Original data
        X_orig_tr = X_orig.iloc[or_tr_idx].copy()
        y_orig_tr = y_orig.iloc[or_tr_idx].reset_index(drop=True)

        # Add original data into train
        X_tr = pd.concat(
            [X_tr, X_orig_tr],
            axis=0
        ).reset_index(drop=True)

        y_tr = pd.concat(
            [y_tr, y_orig_tr],
            axis=0
        ).reset_index(drop=True)

        # Target Encoding
        TE = TargetEncoder(
            cv=5,
            target_type='binary',
            shuffle=True,
            random_state=config.SEED
        )

        tr_enc = TE.fit_transform(X_tr[FEATURES], y_tr)
        val_enc = TE.transform(X_val[FEATURES])
        tst_enc = TE.transform(X_tst[FEATURES])

        X_tr[FEATURES] = tr_enc
        X_val[FEATURES] = val_enc
        X_tst[FEATURES] = tst_enc

        # Model
        model = TabM_D_Classifier(**config.TABM_PARAMS)

        model.fit(
            X_tr,
            y_tr,
            X_val,
            y_val
        )

        # Predictions
        val_preds = model.predict_proba(X_val)[:, 1]
        fold_test_preds = model.predict_proba(X_tst)[:, 1]

        oof_predictions[val_idx] = val_preds
        test_predictions += fold_test_preds / config.N_FOLDS

        # Metrics
        fold_logloss = log_loss(y_val, val_preds)
        fold_auc = roc_auc_score(y_val, val_preds)

        fold_loglosses.append(fold_logloss)
        fold_aucs.append(fold_auc)

        print("=" * 50)
        print(f"Fold {fold} LogLoss : {fold_logloss:.5f}")
        print(f"Fold {fold} ROC AUC : {fold_auc:.5f}")
        print("=" * 50)

    # Overall metrics
    oof_logloss = log_loss(y, oof_predictions)
    oof_auc = roc_auc_score(y, oof_predictions)

    std_oof_logloss = np.std(fold_loglosses)
    std_oof_auc = np.std(fold_aucs)

    mlflow.log_metric("oof_logloss", oof_logloss)
    mlflow.log_metric("oof_auc", oof_auc)
    mlflow.log_metric("std_oof_logloss", std_oof_logloss)
    mlflow.log_metric("std_oof_auc", std_oof_auc)

    mlflow.log_params(config.TABM_PARAMS)

# Saving predictions
oof_proba_df = pd.DataFrame({
    'id': train_ids,
    'PitNextLap': oof_predictions
})

test_proba_df = pd.DataFrame({
    'id': test_ids,
    'PitNextLap': test_predictions
})

save_csv_file(
    oof_proba_df,
    Path(config.OOF_PROBA_DIR, f'{RUN_NAME}_oof_proba.csv')
)

save_csv_file(
    test_proba_df,
    Path(config.TEST_PROBA_DIR, f'{RUN_NAME}_test_proba.csv')
)

Columns classified as continuous: ['driver', 'compound', 'race', 'year', 'pitstop', 'lapnumber', 'stint', 'tyrelife', 'position', 'laptime (s)', 'laptime_delta', 'cumulative_degradation', 'raceprogress', 'position_change', 'Race_Year', 'Race_Compound', '_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber', '_LapNumber_/_RaceProgress_cat', '_TyreLife_/_LapNumber_cat', '_LapTime (s)_*_Cumulative_Degradation', '_LapTime (s)_*_Cumulative_Degradation_abs', '_LapTime (s)_/_Cumulative_Degradation_abs', 'LapTime_Delta', 'driver0', 'LapNumber', 'RaceProgress', 'LapTime (s)_cat', 'LapTime_Delta_cat', 'Cumulative_Degradation_cat', 'year_cat', 'lapnumber_cat']
Columns classified as categorical: []
Device:        CUDA
AMP:           True (dtype: torch.bfloat16)
torch.compile: False
----------------------------------------------------------------------------------------



Epoch 0: 100%|██████████| 421/421 [00:25<00:00, 16.45it/s]


(val) -0.0594
🌸 New best epoch! 🌸



Epoch 1: 100%|██████████| 421/421 [00:24<00:00, 16.85it/s]


(val) -0.0566
🌸 New best epoch! 🌸



...

(val) -0.0534
🌸 New best epoch! 🌸



Epoch 4: 100%|██████████| 421/421 [00:38<00:00, 10.91it/s]


(val) -0.0523
🌸 New best epoch! 🌸



Result:
{'val': -0.05228644609451294, 'epoch': 4}
Restoring best model
Fold 1 LogLoss : 0.22858
Fold 1 ROC AUC : 0.94771
Columns classified as continuous: ['driver', 'compound', 'race', 'year', 'pitstop', 'lapnumber', 'stint', 'tyrelife', 'position', 'laptime (s)', 'laptime_delta', 'cumulative_degradation', 'raceprogress', 'position_change', 'Race_Year', 'Race_Compound', '_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber', '_LapNumber_/_RaceProgress_cat', '_TyreLife_/_LapNumber_cat', '_LapTime (s)_*_Cumulative_Degradation', '_LapTime (s)_*_Cumulative_Degradation_abs', '_LapTime (s)_/_Cumulative_Degradation_abs', 'LapTime_Delta', 'driver0', 'LapNumber', 'RaceProgress', 'LapTime (s)_cat', 'LapTime_Delta_cat', 'Cumulative_Degradation_cat', 'year_cat', 'lapnumber_cat']
Columns classified as categorical: []
Device:        CUDA
AMP:           True (dtype: torch.bfloat16)
torch.compile: False
---------------------------------------------------------------

Epoch 0:  14%|█▍        | 61/421 [00:06<00:35, 10.17it/s]


KeyboardInterrupt: 